# 02 — Model 1: YOLOv8-seg (One-Stage CNN) & Eksperimen NRP 0 (320 / 640 / 960)
**Runtime:** GPU T4 (Colab Free) -> `Runtime > Change runtime type > T4 GPU`.

Notebook ini menjalankan:
1. Training **YOLOv8s-seg** pada 3 ukuran input sesuai **NRP akhiran 0**:
   - `imgsz = 320`
   - `imgsz = 640` (baseline)
   - `imgsz = 960` (batch diperkecil otomatis agar tidak OOM di T4 15GB)
2. Evaluasi setiap model pada **2 test set**:
   - Test set asli (`main_yolo`)
   - Test set buatan sendiri (`custom_hp_yolo`)
3. Pencatatan otomatis **waktu GPU** & penyimpanan `results.csv` asli ke Google Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics pyyaml pandas

import os, time, yaml, json, torch
import pandas as pd
from ultralytics import YOLO

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PROJECT_DIR = '/content/drive/MyDrive/CPMK3_Instance_Segmentation'
MAIN_YAML   = os.path.join(PROJECT_DIR, 'datasets', 'main_yolo', 'data.yaml')
CUSTOM_YAML = os.path.join(PROJECT_DIR, 'datasets', 'custom_hp_yolo', 'data.yaml')
RUNS_DIR    = os.path.join(PROJECT_DIR, 'runs', 'yolov8_seg')
LOGS_DIR    = os.path.join(PROJECT_DIR, 'logs')
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)


## Persiapan `data.yaml` dengan Path Absolut
Memastikan path `train`, `val`, dan `test` di `data.yaml` mengarah ke lokasi absolut di Google Drive.


In [ ]:
def fix_yaml_paths(yaml_path, dataset_root, out_path):
    with open(yaml_path, 'r') as f:
        cfg = yaml.safe_load(f)
    cfg['path'] = dataset_root
    for split in ['train', 'val', 'test']:
        candidate = os.path.join(dataset_root, split, 'images')
        if os.path.exists(candidate):
            cfg[split] = os.path.join(split, 'images')
        elif split == 'val' and os.path.exists(os.path.join(dataset_root, 'valid', 'images')):
            cfg['val'] = os.path.join('valid', 'images')
    # Untuk custom test set, jika semua foto ada di train/images atau test/images:
    if 'test' not in cfg or not os.path.exists(os.path.join(dataset_root, cfg.get('test', ''))):
        for fallback in ['test/images', 'valid/images', 'train/images']:
            if os.path.exists(os.path.join(dataset_root, fallback)):
                cfg['test'] = fallback
                cfg['val'] = fallback
                break
    with open(out_path, 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    print(f"Saved fixed yaml: {out_path}")
    print(cfg)

MAIN_YAML_FIXED   = os.path.join(LOGS_DIR, 'main_yolo_fixed.yaml')
CUSTOM_YAML_FIXED = os.path.join(LOGS_DIR, 'custom_hp_yolo_fixed.yaml')

fix_yaml_paths(MAIN_YAML,   os.path.join(PROJECT_DIR, 'datasets', 'main_yolo'),      MAIN_YAML_FIXED)
fix_yaml_paths(CUSTOM_YAML, os.path.join(PROJECT_DIR, 'datasets', 'custom_hp_yolo'), CUSTOM_YAML_FIXED)


## Eksperimen NRP 0: Training YOLOv8s-seg di Ukuran Input `320`, `640`, `960`
> **Tips Colab Free:** Jika sesi sempat terputus, jalankan ulang cell ini — run yang sudah selesai (`best.pt` sudah ada) otomatis dilewati, atau Anda bisa atur `SIZES_TO_RUN = [320]` satu per satu.


In [ ]:
MODEL_WEIGHTS = 'yolov8s-seg.pt'  # Bisa ganti 'yolov8n-seg.pt' jika ingin lebih cepat
EPOCHS = 50                       # Sesuaikan (30-50 epoch biasanya cukup untuk dataset 3 kelas)

# Pengaturan batch size aman untuk GPU T4 15GB di Colab Free
CONFIGS = {
    320: {'batch': 16, 'workers': 2},
    640: {'batch': 16, 'workers': 2},
    960: {'batch': 4,  'workers': 2},  # Batch kecil di 960 agar tidak OOM
}

SIZES_TO_RUN = [320, 640, 960]
gpu_log_path = os.path.join(LOGS_DIR, 'gpu_time_log.csv')

def append_gpu_log(row_dict):
    df_new = pd.DataFrame([row_dict])
    if os.path.exists(gpu_log_path):
        df_old = pd.read_csv(gpu_log_path)
        df_all = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df_all = df_new
    df_all.to_csv(gpu_log_path, index=False)

for imgsz in SIZES_TO_RUN:
    run_name = f'yolov8s_seg_img{imgsz}'
    best_pt = os.path.join(RUNS_DIR, run_name, 'weights', 'best.pt')
    if os.path.exists(best_pt):
        print(f'[SKIP] {run_name} sudah selesai: {best_pt}')
        continue

    print(f'\n========== TRAINING {run_name} (imgsz={imgsz}) ==========')
    t0 = time.time()
    model = YOLO(MODEL_WEIGHTS)
    model.train(
        data=MAIN_YAML_FIXED,
        epochs=EPOCHS,
        imgsz=imgsz,
        batch=CONFIGS[imgsz]['batch'],
        workers=CONFIGS[imgsz]['workers'],
        project=RUNS_DIR,
        name=run_name,
        exist_ok=True,
        patience=15,
        seed=42,
        amp=True,
        plots=True,
    )
    elapsed_min = round((time.time() - t0) / 60.0, 2)
    print(f'Selesai {run_name} dalam {elapsed_min} menit.')

    append_gpu_log({
        'model': 'YOLOv8s-seg',
        'architecture_family': 'One-Stage CNN',
        'imgsz': imgsz,
        'epochs': EPOCHS,
        'batch_size': CONFIGS[imgsz]['batch'],
        'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'train_time_minutes': elapsed_min,
        'results_csv': os.path.join(RUNS_DIR, run_name, 'results.csv'),
    })
    torch.cuda.empty_cache()


## Evaluasi di 2 Test Set (Test Set Asli vs Test Set HP Sendiri)
Menghitung Box mAP@50, Box mAP@50-95, Mask mAP@50, dan Mask mAP@50-95 untuk analisis **Domain Shift** & **Pengaruh Ukuran Input (NRP 0)**.


In [ ]:
eval_rows = []

for imgsz in [320, 640, 960]:
    run_name = f'yolov8s_seg_img{imgsz}'
    best_pt = os.path.join(RUNS_DIR, run_name, 'weights', 'best.pt')
    if not os.path.exists(best_pt):
        print(f'Belum ada weights untuk {run_name}, lewati.')
        continue

    model = YOLO(best_pt)
    for split_label, yaml_file, split_name in [
        ('Test_Asli', MAIN_YAML_FIXED, 'test'),
        ('Test_HP_Sendiri', CUSTOM_YAML_FIXED, 'test'),
    ]:
        t0 = time.time()
        metrics = model.val(
            data=yaml_file,
            split=split_name,
            imgsz=imgsz,
            batch=CONFIGS[imgsz]['batch'],
            project=os.path.join(RUNS_DIR, 'eval'),
            name=f'{run_name}_{split_label}',
            exist_ok=True,
            plots=True,
        )
        eval_sec = round(time.time() - t0, 2)
        eval_rows.append({
            'model': 'YOLOv8s-seg',
            'architecture_family': 'One-Stage CNN',
            'imgsz': imgsz,
            'dataset': split_label,
            'box_mAP50': round(float(metrics.box.map50), 4),
            'box_mAP50_95': round(float(metrics.box.map), 4),
            'mask_mAP50': round(float(metrics.seg.map50), 4),
            'mask_mAP50_95': round(float(metrics.seg.map), 4),
            'eval_time_sec': eval_sec,
        })

df_yolo_eval = pd.DataFrame(eval_rows)
out_csv = os.path.join(LOGS_DIR, 'eval_yolov8_seg.csv')
df_yolo_eval.to_csv(out_csv, index=False)
print('Hasil evaluasi YOLOv8-seg disimpan ke:', out_csv)
df_yolo_eval
